# Simulate Feedback

Development notebook to simulate implicit user behaviour and inspect how it modifies the Bayesian Network CPDs.

Workflow:
1. Load the BN model and initialise CPT counts
2. Run `simulate_session` calls (individual or batch)
3. Inspect CPD diffs and probability tables after each session

## Setup

In [ ]:
import sys
import os
import copy
import itertools
import pandas as pd

# Make sure main/ is on the path when running from repo root
MODULE_DIR = os.path.dirname(os.path.abspath('__file__'))
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

import feedback as fb
import bn_builder
from simulate_feedback import (
    viewing_to_feedback,
    simulate_session,
    snapshot_counts,
    _counts_to_probs,
    print_cpd_diff,
)

In [ ]:
MODEL_PATH = os.path.join(MODULE_DIR, 'output', 'model.pkl')

print('Loading model ...')
model = bn_builder.load_model(MODEL_PATH)

print('Initialising CPT counts ...')
cpt_counts = fb.initialize_cpt_counts(model)

print('Done. Variables:', list(cpt_counts.keys()))

## Helper: visualise a full CPD as a DataFrame

In [ ]:
def cpd_to_df(cpt_counts, variable):
    """Return a tidy DataFrame with the current probability distribution for *variable*."""
    info = cpt_counts[variable]
    probs = _counts_to_probs(info, variable)
    parents = info['parents']
    var_states = info['state_names'][variable]

    rows = []
    for parent_state, dist in probs.items():
        row = {p: s for p, s in zip(parents, parent_state)} if parents else {}
        row.update(dist)
        rows.append(row)

    df = pd.DataFrame(rows)
    if parents:
        df = df.set_index(parents)
    return df.round(4)


def cpd_diff_df(cpt_before, cpt_after, variable):
    """Return a DataFrame showing the delta (after - before) for each cell."""
    df_before = cpd_to_df(cpt_before, variable)
    df_after  = cpd_to_df(cpt_after,  variable)
    return (df_after - df_before).round(6)

## Viewing → feedback rule explorer

Quickly preview what feedback label and learning rate a given viewing event would produce.

In [ ]:
cases = [
    (100,  1, None),   # full watch, single
    (100,  3, None),   # full watch, rewatch x3
    ( 80,  1, None),   # >70% → accepted
    ( 60,  1, None),   # <70% → rejected
    ( 30,  1, None),   # low %
    ( 80,  1, 4),      # 80% of 4 min = 3.2 min → short-watch guard
    ( 20,  1, 90),     # 20% of 90 min = 18 min → % rule applies
]

rows = []
for pct, times, dur in cases:
    result = viewing_to_feedback(pct, times, dur)
    rows.append({
        'percent_watched': pct,
        'times_watched': times,
        'duration_min': dur,
        'user_feedback': result['user_feedback'],
        'learning_rate': result['learning_rate'],
    })

pd.DataFrame(rows)

## Example 1 – Strong acceptance: watched 100%, senior couple, weekday afternoon

In [ ]:
context_couple_afternoon = {
    'DayType':       'weekday',
    'TimeOfDay':     'afternoon',
    'HouseholdType': 'couple',
    'UserAge':       'senior',
    'UserGender':    'female',
}

before_1 = snapshot_counts(cpt_counts)

simulate_session(
    model, cpt_counts,
    program_type='series',
    program_genre='drama',
    percent_watched=100,
    context_attrs=context_couple_afternoon,
)

In [ ]:
print('--- ProgramType delta ---')
display(cpd_diff_df(before_1, cpt_counts, 'ProgramType'))

print('--- ProgramGenre delta ---')
display(cpd_diff_df(before_1, cpt_counts, 'ProgramGenre'))

## Example 2 – Rejected: watched only 60% (below 70% threshold)

In [ ]:
before_2 = snapshot_counts(cpt_counts)

simulate_session(
    model, cpt_counts,
    program_type='movie',
    program_genre='comedy',
    percent_watched=60,
)

In [ ]:
print('--- ProgramType delta (expect rejection signal) ---')
display(cpd_diff_df(before_2, cpt_counts, 'ProgramType'))

print('--- ProgramGenre delta ---')
display(cpd_diff_df(before_2, cpt_counts, 'ProgramGenre'))

## Example 3 – Rejection: watched 30%, weekend morning

In [ ]:
before_3 = snapshot_counts(cpt_counts)

simulate_session(
    model, cpt_counts,
    program_type='news',
    program_genre='news',
    percent_watched=30,
    context_attrs={'DayType': 'weekend', 'TimeOfDay': 'morning'},
)

In [ ]:
print('--- ProgramType delta ---')
display(cpd_diff_df(before_3, cpt_counts, 'ProgramType'))

print('--- ProgramGenre delta ---')
display(cpd_diff_df(before_3, cpt_counts, 'ProgramGenre'))

## Example 4 – Short-watch guard: 10% of 90 min → strong rejection

In [ ]:
before_4 = snapshot_counts(cpt_counts)

simulate_session(
    model, cpt_counts,
    program_type='documentary',
    program_genre='documentary',
    percent_watched=10,
    duration_minutes=90,
)

In [ ]:
print('--- ProgramType delta ---')
display(cpd_diff_df(before_4, cpt_counts, 'ProgramType'))

print('--- ProgramGenre delta ---')
display(cpd_diff_df(before_4, cpt_counts, 'ProgramGenre'))

## Example 5 – Rewatch x3: learning_rate tripled (strong positive signal)

In [ ]:
before_5 = snapshot_counts(cpt_counts)

simulate_session(
    model, cpt_counts,
    program_type='movie',
    program_genre='romance',
    percent_watched=100,
    times_watched=3,
    context_attrs=context_couple_afternoon,
)

In [ ]:
print('--- ProgramType delta ---')
display(cpd_diff_df(before_5, cpt_counts, 'ProgramType'))

print('--- ProgramGenre delta ---')
display(cpd_diff_df(before_5, cpt_counts, 'ProgramGenre'))

## Sandbox – custom session

Edit the parameters below and re-run to test any combination.

In [ ]:
# --- edit these ---
PROGRAM_TYPE    = 'series'
PROGRAM_GENRE   = 'thriller'
PERCENT_WATCHED = 85
TIMES_WATCHED   = 1
DURATION_MIN    = None
CONTEXT = {
    'DayType':   'weekday',
    'TimeOfDay': 'night',
    # 'UserAge':       'adult',
    # 'UserGender':    'male',
    # 'HouseholdType': 'single',
}
# ------------------

before_custom = snapshot_counts(cpt_counts)

simulate_session(
    model, cpt_counts,
    program_type=PROGRAM_TYPE,
    program_genre=PROGRAM_GENRE,
    percent_watched=PERCENT_WATCHED,
    times_watched=TIMES_WATCHED,
    duration_minutes=DURATION_MIN,
    context_attrs=CONTEXT,
)

In [ ]:
for var in ['ProgramType', 'ProgramGenre']:
    delta = cpd_diff_df(before_custom, cpt_counts, var)
    changed = delta[(delta.abs() > 1e-8).any(axis=1)]
    print(f'--- {var} delta (changed rows only) ---')
    display(changed if not changed.empty else '(no changes)')

print('\n--- Full ProgramType CPD (current state) ---')
display(cpd_to_df(cpt_counts, 'ProgramType'))

print('\n--- Full ProgramGenre CPD (current state) ---')
display(cpd_to_df(cpt_counts, 'ProgramGenre'))